### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="emscad",
    dataset_year="2014",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction/data",
    download_description="""
The original website (http://icsdweb.aegean.gr/emscad) has been down for a while.
Thus, we us an alternative source from Kaggle (https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction/data).
There is also an additional alternative source from Kaggle (https://www.kaggle.com/datasets/amruthjithrajvr/recruitment-scam).

kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction -f fake_job_postings.csv && unzip fake_job_postings.csv.zip fake_job_postings.csv&& rm fake_job_postings.csv.zip
mkdir -p local-data-warehouse/emscad && mv fake_job_postings.csv local-data-warehouse/emscad
""",
    # References
    academic_reference_bibtex="""@article{vidros2017automatic,
  title={Automatic detection of online recruitment frauds: Characteristics, methods, and a public dataset},
  author={Vidros, Sokratis and Kolias, Constantinos and Kambourakis, Georgios and Akoglu, Leman},
  journal={Future Internet},
  volume={9},
  number={1},
  pages={6},
  year={2017},
  publisher={MDPI}
}
""",
    academic_reference_bibtex_key="vidros2017automatic",
    license="CC0: Public Domain", # from Kaggle, paper does not state anything, website is down to check -> so maybe None...
    data_tags=["IID", "Spatial"],
    curation_comments="""
We have no access to the original state and the data already contains some preprocessed features.
The paper describes them in more detail as well in Table 2.

In the original paper, the authors subsampled the data to have 450 fraudulent and 450 non-fraudulent samples.
No details are given on how the subsampling was done. Moreover, duplicates were skipped, but no details are given on how duplicates were identified.

- The dataset was collected from 2012 to 2014. But the data contains no feature to identify the time of a sample. So we can expect that some bias from temporal leakage. Moreover, the data might contain multiple job postings from the same fraudster, which would need to be grouped, but we cannot identify them from the data.
- Following the original paper, we drop all duplicates (when ignoring the job_id column) to avoid data leakage.
- The data has a location description, we do not resolve it lat/longitude but leave it to the pipelines.
- The salary range has several weird quirks. It contains data either in the thousands, or is missing the "k" to denote thousands. Moreover, it contains empty or 0-0 entries. The column might contain yearly salary, hourly salary, or one-time payment. Finally, for some jobs, the column contains a date (e.g. "Oct-20"). We keep the column as complex as it is. We add one column that contains the maximum salary parsed from the text and we only keep salary values above 50k as valid values to create a numerical feature for differences in the larger ranges.
- We removed job listings with non english texts (138 rows).
- We do not subsample the data, as a result, the data is heavily imbalanced.
- We found no relation of job_id with a timestamp and even found cases where a lower job_id has a description claiming to be a job from 2014 (so the end of the collection period). Thus, we do dropped job_id as it does not contain any signal.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="fraudulent",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="fraudulent",
)

## Preprocessing

In [ ]:
import pandas as pd
import  numpy as np
df = pd.read_csv(dataset_mold.path / "fake_job_postings.csv")
print("Loaded data shape:", df.shape)

# Drop duplicates
df = df[~df.drop(columns=["job_id"]).duplicated(keep="last")]

# Get max salary proxy for larger ranges
s = df["salary_range"].astype(str).str.strip()
invalid_mask = (df["salary_range"].isna() | (s == '') | (s == '0-0'))
ranges = s.str.split('-', expand=True)
min_vals, max_vals = ranges[0], ranges[1]
is_not_salary_range = pd.to_numeric(max_vals, errors="coerce").isna() | pd.to_numeric(min_vals, errors="coerce").isna() # remove dates
not_above_50_k =  ~(pd.to_numeric(max_vals, errors="coerce") >= 50_000)
df["salary_range_max"] = pd.to_numeric(max_vals, errors="coerce")
df.loc[invalid_mask | is_not_salary_range |not_above_50_k, "salary_range_max"] = np.nan

# from langdetect import detect, LangDetectException
#
# def is_not_english(text: str) -> bool:
#     try:
#         return detect(text) != "en"
#     except LangDetectException:
#         # Raised for very short or ambiguous text
#         return False
# df["is_english_description"] = df["description"].astype(str).apply(lambda x: not is_not_english(x))
# bad_filter_list = [
#     # Missing, faulty or, Lorem ipsum description...
#     1243,
#     3031,
#     7613,
#     5557,
#     11078,
#     11894,
#     12190,
#     13530,
# ]
# print(list(df[(~df["is_english_description"]) & (~df["description"].isna()) & (df["description"] != "Sales Executive") & (~df["job_id"].isin(bad_filter_list))]["job_id"]))
# Output from the above
filter_non_en_job = [543, 550, 557, 687, 936, 986, 1174, 1558, 1667, 1689, 1794, 1831, 1922, 2351, 2384, 2424, 2474, 2677, 2788, 2857, 2929, 2967, 2986, 3126, 3132, 3139, 3156, 3169, 3566, 3576, 3702, 3764, 3856, 3952, 3993, 4157, 4185, 4193, 4315, 4362, 4371, 4401, 4812, 4825, 4867, 5058, 5198, 5274, 5632, 5637, 5651, 5693, 5724, 5931, 6023, 6096, 6169, 6184, 6269, 6284, 6438, 6445, 6573, 6587, 6699, 6752, 6853, 6988, 7025, 7112, 7212, 7549, 7679, 7849, 7881, 8066, 8106, 8214, 8355, 8365, 8397, 8615, 8719, 8791, 8895, 9216, 9393, 9413, 9965, 10450, 10468, 10557, 10570, 10661, 11075, 11314, 11320, 11383, 11419, 11424, 11426, 11427, 11446, 11449, 11463, 11464, 11536, 11631, 11748, 11958, 12231, 12395, 12683, 12973, 13016, 13301, 13474, 13476, 13551, 13696, 14021, 14131, 14333, 14911, 15185, 15410, 16343, 16500, 16586, 16608, 16691, 17034, 17047, 17148, 17226, 17328, 17357, 17781]
df = df[~df["job_id"].isin(filter_non_en_job)]

as_string_col = [
    "title",
    "location",
    "department",
    "salary_range",
    # HTML fragments
    "company_profile",
    "description",
    "requirements",
    "benefits",
]
as_cat_type = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "fraudulent",
]

for c in as_string_col:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["job_id"]).reset_index(drop=True)

# After our preprocessing, there is just one entry without a description, so we drop it as well.
df = df[~df["description"].isna()].reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_splits_dimensions, get_recommended_iid_splits

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="In the original paper, 10-fold CV is used. We follow our default suggestions in the case of IID.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()